# Tankerkönig API Test — 10 closest stations from Venlo

In [ ]:
import requests
import math
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────────
API_KEY   = "201eeca2-d06c-4adc-ab5b-5bec8f5faf27"
FUEL_TYPE = "e5"        # e5 | e10 | diesel
RADIUS_KM = 20          # per query point (Tankerkönig max = 25)
TOP_N     = 10

# Venlo, Netherlands
USER_LAT, USER_LNG = 51.3704, 6.1724

# Dutch-German border crossing points (north → south)
BORDER_POINTS = [
    ("Bunde",          53.180, 7.230),
    ("Nieuweschans",   53.190, 7.060),
    ("Ter Apel",       52.880, 7.070),
    ("Coevorden",      52.660, 6.750),
    ("Denekamp",       52.380, 7.020),
    ("Oldenzaal",      52.310, 7.000),
    ("Gronau",         52.210, 7.020),
    ("Bad Bentheim",   52.300, 7.160),
    ("Goch",           51.680, 6.160),
    ("Kleve",          51.790, 6.140),
    ("Emmerich",       51.830, 6.250),
    ("Venlo",          51.370, 6.170),
    ("Kaldenkirchen",  51.320, 6.200),
    ("Roermond-DE",    51.200, 6.100),
    ("Aachen",         50.780, 6.080),
    ("Vaals",          50.770, 6.020),
]

print(f"Config: fuel={FUEL_TYPE}, user=({USER_LAT}, {USER_LNG}), radius={RADIUS_KM} km")


In [ ]:
def haversine_km(lat1, lng1, lat2, lng2):
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlng = math.radians(lng2 - lng1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlng/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def fetch_stations(lat, lng, radius, fuel_type):
    url = (
        f"https://creativecommons.tankerkoenig.de/json/list.php"
        f"?lat={lat}&lng={lng}&rad={radius}&sort=dist&type={fuel_type}&apikey={API_KEY}"
    )
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        data = r.json()
        if not data.get("ok"):
            print(f"  API error: {data.get('message', 'unknown')}")
            return []
        return data.get("stations", [])
    except requests.exceptions.Timeout:
        print(f"  ⏱ Timeout for ({lat}, {lng})")
        return []
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return []


# Only query points within 200 km of user
nearby = [(name, lat, lng) for name, lat, lng in BORDER_POINTS
          if haversine_km(USER_LAT, USER_LNG, lat, lng) < 200]

print(f"Querying {len(nearby)} border points within 200 km of user...\n")

all_stations = {}
for name, lat, lng in nearby:
    dist_to_point = haversine_km(USER_LAT, USER_LNG, lat, lng)
    print(f"  → {name} ({dist_to_point:.0f} km from user)", end=" ")
    stations = fetch_stations(lat, lng, RADIUS_KM, FUEL_TYPE)
    new = 0
    for s in stations:
        if s["id"] not in all_stations:
            # Recalculate distance from USER, not from border point
            s["dist_from_user"] = haversine_km(USER_LAT, USER_LNG, s["lat"], s["lng"])
            all_stations[s["id"]] = s
            new += 1
    print(f"→ {len(stations)} stations ({new} new)")

print(f"\nTotal unique German stations found: {len(all_stations)}")


In [ ]:
# Sort by distance from user, show top N
sorted_stations = sorted(all_stations.values(), key=lambda s: s["dist_from_user"])
top = sorted_stations[:TOP_N]

rows = []
for i, s in enumerate(top, 1):
    price = s.get(FUEL_TYPE)
    rows.append({
        "rank":    i,
        "brand":   s.get("brand") or s.get("name"),
        "name":    s.get("name"),
        "place":   s.get("place"),
        "street":  f"{s.get('street','')} {s.get('houseNumber','')}".strip(),
        f"{FUEL_TYPE} (€/L)": f"{price:.3f}" if price else "—",
        "dist_km": round(s["dist_from_user"], 1),
        "isOpen":  "✓" if s.get("isOpen") else "✗",
    })

df = pd.DataFrame(rows).set_index("rank")
print(f"\n🏆 Top {TOP_N} closest German stations from Venlo ({FUEL_TYPE.upper()})\n")
print(df.to_string())
